# Calling the Databricks budgets REST API: three read-only approaches

This notebook compares ways to call the account-level **Budgets** API and includes one workspace API control call that should succeed for any authenticated workspace user.

| # | Approach | Host | Credential | Same identity as the workspace login? |
|---|----------|------|------------|----------------------------------------|
| Control | Raw `requests` GET `/api/2.0/preview/scim/v2/Me` | Workspace | Workspace OAuth/PAT token from `WorkspaceClient` | Yes |
| 1 | Raw `requests` GET `/api/2.1/accounts/{account_id}/budgets` reusing the workspace token | Account console | Same workspace token | Yes, but wrong token audience |
| 2 | SDK `AccountClient.budgets.list()` | Account console | Account-level U2M OAuth token for the same user (`databricks auth login --host https://accounts.cloud.databricks.com --account-id ...`) | Yes, separate login |
| 3 | CLI `databricks account budgets list` / `databricks api get` | Account console | An account-level CLI profile | Whoever the profile authenticates |

Endpoint paths come from the official API reference: [Budgets: list](https://docs.databricks.com/api/account/budgets/list) is `GET /api/2.1/accounts/{account_id}/budgets` with optional `page_token`. Budgets are account-level only; there is no documented workspace-host budgets route, so this notebook does not try one. Budget *policies* (`/api/2.1/accounts/{account_id}/budget-policies`) are a different API and are out of scope.

Approaches 1 and 2 differ in **credential**, not just transport. Approaches 2 and 3 use the same credential type and differ only in transport (Python SDK versus CLI). Per the [OAuth U2M docs](https://docs.databricks.com/aws/en/dev-tools/auth/oauth-u2m), a workspace-level token can call only that workspace's REST APIs; account APIs need an account-level login against `https://accounts.cloud.databricks.com`.

Every call here is a `GET` with a finite timeout. Output shows status codes, error codes, counts and masked identifiers only, never tokens, headers, or full principal or resource lists.

## Saved outputs in this copy

The outputs saved below come from a local Jupyter run on 2026-10-19 with workspace profile `DEFAULT` (`databricks-cli` U2M auth), with `DATABRICKS_CONFIG_PROFILE` unset and `DATABRICKS_ACCOUNT_PROFILE` unset:

- **Control:** executed, HTTP 200.
- **Approach 1:** executed, HTTP 400 `Unable to load OAuth Config`. That's a token-audience mismatch, not evidence about admin rights.
- **Approach 2:** executed, no account-level login cached for this user, so the SDK couldn't build credentials and no request was sent.
- **Approach 3:** **not executed**, because no account-level profile was selected and none was created.

Re-run the notebook to replace these outputs with your own.

## Running locally versus in Databricks

**Local Jupyter.** Authentication comes from `~/.databrickscfg`. Set `DATABRICKS_WORKSPACE_PROFILE` to choose a workspace profile (defaults to `DEFAULT`). Leave `DATABRICKS_CONFIG_PROFILE` unset: the SDK merges that profile into every client it builds, including the account client, and the account call would then quietly reuse the workspace token. `DATABRICKS_ACCOUNT_ID` overrides the `account_id` read from that profile. Approach 2 uses the Databricks CLI token cache for the account host, so it only succeeds after a one-time interactive `databricks auth login --host https://accounts.cloud.databricks.com --account-id <id>`; the notebook never starts that login itself. Approach 3 runs only if you set `DATABRICKS_ACCOUNT_PROFILE` to an existing account-level profile name.

**Databricks notebook.** The notebook's native credential is a workspace-scoped token, so the control call and approach 1 run with no setup. The same settings are exposed as widgets. There is no CLI token cache on the cluster, so the same-user account login used by approaches 2 and 3 isn't available there. Calling account APIs from a job or notebook usually means an account-level service principal (OAuth M2M with a client secret stored in a secret scope). That is a different identity, so it isn't shown here.

Install requirements locally with `pip install databricks-sdk requests`. Both are preinstalled in Databricks Runtime.

## Configuration and helpers

Settings resolve from widgets when running in Databricks, then from environment variables, then from defaults. The helpers mask emails, hosts, and IDs, and remove anything shaped like a bearer token or JWT from error text before printing.

In [1]:
import itertools
import json
import os
import re
import shutil
import subprocess
from urllib.parse import urlparse

import requests
from databricks.sdk import AccountClient, WorkspaceClient
from databricks.sdk.core import Config

IN_DATABRICKS_RUNTIME = "DATABRICKS_RUNTIME_VERSION" in os.environ


def setting(name, default=""):
    if IN_DATABRICKS_RUNTIME:
        dbutils.widgets.text(name, default)
        value = dbutils.widgets.get(name).strip()
        if value:
            return value
    return os.environ.get(name, default).strip()


PROFILE = "" if IN_DATABRICKS_RUNTIME else setting("DATABRICKS_WORKSPACE_PROFILE", "DEFAULT")
ACCOUNTS_HOST = setting("DATABRICKS_ACCOUNTS_HOST", "https://accounts.cloud.databricks.com").rstrip("/")
ACCOUNT_PROFILE = setting("DATABRICKS_ACCOUNT_PROFILE")
HTTP_TIMEOUT_SECONDS = int(setting("HTTP_TIMEOUT_SECONDS", "30"))
MAX_BUDGETS_TO_COUNT = 500

SECRET_PATTERNS = [
    re.compile(r"(?i)bearer\s+\S+"),
    re.compile(r"\beyJ[\w-]+\.[\w-]+\.[\w-]*"),
    re.compile(r"\bdap[a-z]?[0-9a-f]{20,}\S*"),
    re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+"),
    re.compile(r"\b[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}\b"),
]


def scrub(text, limit=300):
    text = str(text)
    for pattern in SECRET_PATTERNS:
        text = pattern.sub("<redacted>", text)
    return text if len(text) <= limit else text[:limit] + "..."


def mask_id(value, keep=4):
    value = str(value or "")
    return f"...{value[-keep:]}" if value else "<unset>"


def mask_email(value):
    value = str(value or "")
    if "@" not in value:
        return mask_id(value)
    name, domain = value.split("@", 1)
    return f"{name[:1]}***@{domain}"


def mask_host(url):
    host = urlparse(url or "").hostname or ""
    if host == urlparse(ACCOUNTS_HOST).hostname:
        return host
    labels = host.split(".")
    return "*." + ".".join(labels[-3:]) if len(labels) > 3 else host


def show(result):
    print(json.dumps(result, indent=2, default=str))


def summarize_http(label, response, list_key=None):
    result = {"call": label, "http_status": response.status_code}
    try:
        body = response.json()
    except ValueError:
        body = {}
        result["body_excerpt"] = scrub(response.text)
    if isinstance(body, dict):
        if body.get("error_code") or body.get("message") or body.get("error"):
            result["error_code"] = body.get("error_code") or body.get("error")
            result["message"] = scrub(body.get("message") or body.get("error_description") or "")
        if list_key and response.ok:
            result[f"{list_key}_returned"] = len(body.get(list_key) or [])
            result["has_next_page"] = bool(body.get("next_page_token"))
    return result


def workspace_client():
    kwargs = {"http_timeout_seconds": HTTP_TIMEOUT_SECONDS, "retry_timeout_seconds": HTTP_TIMEOUT_SECONDS * 2}
    if PROFILE:
        kwargs["profile"] = PROFILE
    return WorkspaceClient(config=Config(**kwargs))


w = workspace_client()
ACCOUNT_ID = setting("DATABRICKS_ACCOUNT_ID") or (w.config.account_id or "")
BUDGETS_PATH = f"/api/2.1/accounts/{ACCOUNT_ID}/budgets"

show({
    "runtime": "databricks" if IN_DATABRICKS_RUNTIME else "local",
    "workspace_profile": PROFILE or "<notebook default auth>",
    "workspace_auth_type": w.config.auth_type,
    "workspace_host": mask_host(w.config.host),
    "accounts_host": ACCOUNTS_HOST,
    "account_id": mask_id(ACCOUNT_ID),
    "account_profile": ACCOUNT_PROFILE or "<unset>",
    "timeout_seconds": HTTP_TIMEOUT_SECONDS,
})

{
  "runtime": "local",
  "workspace_profile": "DEFAULT",
  "workspace_auth_type": "databricks-cli",
  "workspace_host": "*.cloud.databricks.com",
  "accounts_host": "https://accounts.cloud.databricks.com",
  "account_id": "...0ec0",
  "account_profile": "<unset>",
  "timeout_seconds": 30
}


## Control: a workspace API call that should succeed

`GET /api/2.0/preview/scim/v2/Me` ([docs](https://docs.databricks.com/api/workspace/currentuser/me)) returns the calling user and is available to every workspace user. It uses the same `requests` transport and the same workspace token as approach 1, so if this returns 200 and approach 1 fails, the difference is the target host and token audience, not the HTTP client.

`w.config.authenticate()` returns ready-to-send auth headers for whatever credential the SDK resolved: the CLI OAuth cache locally, or the notebook token in Databricks. The headers are never printed.

In [2]:
workspace_headers = w.config.authenticate()
me_response = requests.get(
    f"{w.config.host}/api/2.0/preview/scim/v2/Me",
    headers=workspace_headers,
    timeout=HTTP_TIMEOUT_SECONDS,
)
control = summarize_http("GET /api/2.0/preview/scim/v2/Me", me_response)
if me_response.ok:
    me = me_response.json()
    group_names = {group.get("display") for group in me.get("groups", [])}
    control.update({
        "user": mask_email(me.get("userName")),
        "active": me.get("active"),
        "group_count": len(group_names),
        "in_workspace_admins_group": "admins" in group_names,
    })
show(control)

{
  "call": "GET /api/2.0/preview/scim/v2/Me",
  "http_status": 200,
  "user": "j***@databricks.com",
  "active": true,
  "group_count": 4,
  "in_workspace_admins_group": true
}


## Approach 1: raw REST to the account host with the workspace token

This sends the workspace token from the control call to the documented account endpoint. Expect it to fail. A workspace-issued OAuth token is for the workspace host, and the account console doesn't accept it. A `400` with `Unable to load OAuth Config`, or a `401`, means the credential is wrong for this host. It **doesn't** tell you whether the user is an account admin or has billing access.

In [3]:
if not ACCOUNT_ID:
    print("Not executed: set DATABRICKS_ACCOUNT_ID (env var or widget).")
else:
    approach_1 = requests.get(
        f"{ACCOUNTS_HOST}{BUDGETS_PATH}",
        headers=workspace_headers,
        timeout=HTTP_TIMEOUT_SECONDS,
    )
    show(summarize_http("GET /api/2.1/accounts/{account_id}/budgets (workspace token)", approach_1, "budgets"))

{
  "call": "GET /api/2.1/accounts/{account_id}/budgets (workspace token)",
  "http_status": 400,
  "body_excerpt": "Unable to load OAuth Config"
}


## Approach 2: SDK `AccountClient` with an account-level login for the same user

`AccountClient` configured with `host=ACCOUNTS_HOST`, `account_id=ACCOUNT_ID` and `auth_type="databricks-cli"` asks the Databricks CLI for a cached U2M token for the account host. That token is for the same human user as the workspace login, but it comes from a separate OAuth login against the account console, so it has a different audience. Pagination is handled by `budgets.list()`.

Prerequisite (interactive, one time, run in a terminal): `databricks auth login --host https://accounts.cloud.databricks.com --account-id <account-id>`. If `DATABRICKS_ACCOUNT_PROFILE` is set, that profile is used instead. Nothing here falls back to another profile automatically.

If there's no cached account token, the cell reports the SDK error (`cannot configure default credentials`) and stops; it doesn't open a browser. The cell refuses to run while `DATABRICKS_CONFIG_PROFILE` is set, because the SDK would merge that workspace profile into the account config and `databricks auth token --profile ...` would return the workspace token, turning this back into approach 1. Once you're authenticated, a `403 PERMISSION_DENIED` is the response that actually means missing account-admin or billing permissions.

In [4]:
def account_client():
    timeouts = {"http_timeout_seconds": HTTP_TIMEOUT_SECONDS, "retry_timeout_seconds": HTTP_TIMEOUT_SECONDS * 2}
    if ACCOUNT_PROFILE:
        return AccountClient(config=Config(profile=ACCOUNT_PROFILE, **timeouts))
    return AccountClient(config=Config(host=ACCOUNTS_HOST, account_id=ACCOUNT_ID, auth_type="databricks-cli", **timeouts))


approach_2 = {"call": "AccountClient.budgets.list()", "credential": ACCOUNT_PROFILE or "databricks-cli U2M cache for accounts host"}
if not (ACCOUNT_ID or ACCOUNT_PROFILE):
    approach_2["result"] = "Not executed: set DATABRICKS_ACCOUNT_ID or DATABRICKS_ACCOUNT_PROFILE."
elif os.environ.get("DATABRICKS_CONFIG_PROFILE") and not ACCOUNT_PROFILE:
    approach_2["result"] = "Not executed: unset DATABRICKS_CONFIG_PROFILE, which would make the account client reuse a workspace profile."
else:
    try:
        a = account_client()
        approach_2["resolved_host"] = mask_host(a.config.host)
        approach_2["resolved_profile"] = a.config.profile or "<none>"
        approach_2["budgets_counted"] = sum(1 for _ in itertools.islice(a.budgets.list(), MAX_BUDGETS_TO_COUNT))
        approach_2["result"] = "success"
    except Exception as exc:
        approach_2["result"] = "error"
        approach_2["error_type"] = type(exc).__name__
        approach_2["error_code"] = getattr(exc, "error_code", None)
        approach_2["message"] = scrub(exc)
show(approach_2)

{
  "call": "AccountClient.budgets.list()",
  "credential": "databricks-cli U2M cache for accounts host",
  "result": "error",
  "error_type": "ValueError",
  "error_code": null,
  "message": "default auth: cannot configure default credentials, please check https://docs.databricks.com/en/dev-tools/auth.html#databricks-client-unified-authentication to configure credentials for your preferred authentication method. Config: host=https://accounts.cloud.databricks.com, account_id=<redacted>, w..."
}


## Approach 3: Databricks CLI against an account-level profile

This uses the same account-level credential as approach 2 through the CLI instead of the Python SDK. `databricks account budgets list` is the typed command. `databricks api get` makes the raw REST call to the documented path and is useful for comparing results with approach 1. Both need an **existing** account-level profile (a `~/.databrickscfg` entry with `host = https://accounts.cloud.databricks.com` and `account_id`). The cell runs only when `DATABRICKS_ACCOUNT_PROFILE` is set, and it reports counts only.

The CLI isn't installed on Databricks clusters by default, so this approach is meant for local use.

In [5]:
def run_cli(args):
    try:
        completed = subprocess.run(args, capture_output=True, text=True, timeout=HTTP_TIMEOUT_SECONDS * 2)
    except subprocess.TimeoutExpired:
        return {"command": " ".join(args[1:3]), "result": "timeout"}
    result = {"command": " ".join(args[1:4]), "exit_code": completed.returncode}
    if completed.returncode != 0:
        result["stderr"] = scrub(completed.stderr)
        return result
    payload = json.loads(completed.stdout or "null")
    budgets = payload.get("budgets", []) if isinstance(payload, dict) else payload or []
    result["budgets_returned"] = len(budgets)
    return result


cli = shutil.which("databricks")
if not ACCOUNT_PROFILE:
    print("Not executed: set DATABRICKS_ACCOUNT_PROFILE to an existing account-level CLI profile.")
elif not cli:
    print("Not executed: Databricks CLI not found on PATH.")
else:
    show(run_cli([cli, "account", "budgets", "list", "--profile", ACCOUNT_PROFILE, "--output", "json"]))
    show(run_cli([cli, "api", "get", BUDGETS_PATH, "--profile", ACCOUNT_PROFILE]))

Not executed: set DATABRICKS_ACCOUNT_PROFILE to an existing account-level CLI profile.


## Reading the results

| Observation | Meaning |
|-------------|---------|
| Control 200, approach 1 `400 Unable to load OAuth Config` or `401` | The workspace token is valid but has the wrong audience for the account host. It says nothing about permissions. |
| Approach 2 error that mentions the token cache or `databricks auth login` | No account-level login has been done for this user and account yet. |
| Approach 2 or 3 `403 PERMISSION_DENIED` | The account-level login works, but the user isn't allowed to read budgets (account admin or billing admin is typically required). |
| Approach 2 or 3 success with a count | The account login and permissions are both sufficient. |

Workspace admin status (shown by `in_workspace_admins_group` in the control) doesn't grant account-level access.